# IMS bearing data exploration

This notebook explores the IMS bearing run-to-failure files when they are available. The IMS dataset is optional and is not included in the initial repository data. To download it, run `python scripts/download_datasets.py --dataset ims` from the repository root. The download may take a while.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data").exists():
    ROOT = ROOT.parent
ims_dir = ROOT / "data" / "raw" / "ims_bearings"


In [ ]:
if not ims_dir.exists():
    print(f"IMS data not found at {ims_dir}. Follow the download instructions above to enable this analysis.")
else:
    files = sorted(path for path in ims_dir.rglob("*") if path.is_file() and not path.name.startswith("."))
    print(f"Found {len(files)} candidate data files under {ims_dir}")
    print("First files:", [path.name for path in files[:5]])


In [ ]:
if ims_dir.exists() and files:
    sample_file = files[0]
    sample = pd.read_csv(sample_file, sep=r"\s+", header=None)
    print(f"Sample file: {sample_file.name}; shape: {sample.shape}")
    display(sample.head())

    channel = sample.iloc[:, 0].dropna().to_numpy()
    plt.figure(figsize=(10, 4))
    plt.plot(channel[: min(2000, len(channel))], linewidth=0.7)
    plt.xlabel("Sample")
    plt.ylabel("Amplitude (channel 1)")
    plt.title(f"IMS waveform excerpt: {sample_file.name}")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
if ims_dir.exists() and files:
    records = []
    for path in files[:30]:
        frame = pd.read_csv(path, sep=r"\s+", header=None)
        signal = frame.iloc[:, 0].dropna().to_numpy(dtype=float)
        if signal.size:
            records.append({"file": path.name, "channel_1_rms": float(np.sqrt(np.mean(signal ** 2)))})
    rms = pd.DataFrame(records)
    display(rms.head())
    if not rms.empty:
        plt.figure(figsize=(10, 4))
        plt.plot(np.arange(len(rms)), rms["channel_1_rms"], marker=".")
        plt.xlabel("File index (chronological filename order)")
        plt.ylabel("Channel 1 RMS")
        plt.title("IMS bearing signal energy across the first 30 files")
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.show()
